# Dual-Task Interference — Concurrent Task Performance

**Track: Attention** | Can the model perform two tasks simultaneously without interference?

Humans show robust dual-task costs when performing two attention-demanding tasks
concurrently. This benchmark measures whether LLMs show analogous interference by
requiring simultaneous comprehension (answer questions) and monitoring (count targets)
within the same passage.

**Metrics:**
- Primary task accuracy (comprehension) under single vs. dual conditions
- Secondary task accuracy (counting) under single vs. dual conditions
- Dual-task cost for each task
- Asymmetric interference (which task suffers more)

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import numpy as np
import json
import re
import random

print(f"Available models: {list(kbench.llms.keys())}")

In [ ]:
def strip_thinking(text: str) -> str:
    if "</think>" in text:
        return text.split("</think>", 1)[1].strip()
    return text.strip()

def extract_count(response: str) -> int | None:
    """Extract a count number from the response."""
    response = strip_thinking(response)
    # Look for "Count: N" or "count: N" pattern
    match = re.search(r"count\s*[:=]\s*(\d+)", response, re.IGNORECASE)
    if match:
        return int(match.group(1))
    # Look for "N occurrences" or "N times"
    match = re.search(r"(\d+)\s*(?:occurrences?|times?|instances?)", response, re.IGNORECASE)
    if match:
        return int(match.group(1))
    # Last number in response
    numbers = re.findall(r"\d+", response)
    return int(numbers[-1]) if numbers else None

def extract_answer(response: str, key: str = "Answer") -> str:
    """Extract an answer value from key: value format."""
    response = strip_thinking(response)
    match = re.search(rf"{re.escape(key)}\s*[:=]\s*(.+)", response, re.IGNORECASE)
    if match:
        return match.group(1).strip().strip('"').strip("'")
    # Fallback: last line
    lines = [l.strip() for l in response.strip().split("\n") if l.strip()]
    return lines[-1] if lines else "" 

In [ ]:
# --- Passages with comprehension questions + counting targets ---

PASSAGE_TEMPLATES = [
    {
        "id": "fusion",
        "text": "The ITER fusion reactor project in southern France represents humanity's most ambitious energy experiment. The tokamak chamber measures {measure} meters in diameter and weighs {weight} tonnes. Engineers from {countries} nations collaborate to achieve sustained plasma temperatures exceeding {temp} million degrees Celsius. The superconducting magnets require cooling to {cool_temp} degrees above absolute zero using liquid helium circuits spanning {pipe_km} kilometers of pipe. Project director Dr. {director} estimates first full-power operations will begin in {year}. The facility's total construction cost has reached {cost} billion euros, making it the most expensive scientific instrument ever built. The deuterium-tritium fuel cycle produces {output} megawatts of thermal energy from just {input} grams of fuel per hour.",
        "params": {"measure": "30", "weight": "23,000", "countries": "35", "temp": "150", "cool_temp": "4", "pipe_km": "200", "director": "Nakamura", "year": "2035", "cost": "22", "output": "500", "input": "0.5"},
        "questions": [
            {"q": "How many nations collaborate on ITER?", "a": "35"},
            {"q": "What is the target plasma temperature?", "a": "150 million degrees"},
            {"q": "What year are full-power operations expected?", "a": "2035"},
            {"q": "What is the total construction cost?", "a": "22 billion euros"},
        ],
    },
    {
        "id": "ocean",
        "text": "The Pacific Garbage Patch extends across {area} million square kilometers between Hawaii and California. Marine biologist Dr. {scientist} estimates it contains {tonnes} million tonnes of plastic debris, with {micro_pct} percent consisting of microplastics smaller than {size} millimeters. Cleanup vessel {vessel} uses a {barrier_len}-meter floating barrier to collect surface debris at a rate of {rate} tonnes per day. The project costs {daily_cost} dollars per day to operate. Deep-water surveys reveal plastic fragments at depths of {depth} meters, far below the reach of surface collection systems. Chemical analysis shows {pct_bottles} percent of recovered material originates from single-use beverage containers manufactured across {mfg_countries} countries.",
        "params": {"area": "1.6", "scientist": "Vasquez", "tonnes": "80", "micro_pct": "94", "size": "5", "vessel": "Ocean Pioneer", "barrier_len": "600", "rate": "50", "daily_cost": "140,000", "depth": "4,200", "pct_bottles": "46", "mfg_countries": "23"},
        "questions": [
            {"q": "What percentage of debris is microplastics?", "a": "94 percent"},
            {"q": "How long is the floating barrier?", "a": "600 meters"},
            {"q": "At what depth have plastic fragments been found?", "a": "4,200 meters"},
            {"q": "What percentage comes from beverage containers?", "a": "46 percent"},
        ],
    },
    {
        "id": "quantum",
        "text": "IBM's Condor quantum processor contains {qubits} superconducting qubits arranged in a {topology} topology. The chip operates at {temp} millikelvin inside a {stage}-stage dilution refrigerator weighing {fridge_weight} tonnes. Error correction uses the {code} surface code requiring {physical_per} physical qubits per logical qubit. Chief architect Dr. {architect} reports gate fidelities of {fidelity} percent for two-qubit operations. The system performs {ops} billion quantum operations per second with a coherence time of {coherence} microseconds. Benchmarking against classical supercomputers showed a {speedup}x advantage on optimization problems involving {variables} variables. The project required {engineers} engineers working over {dev_years} years of development.",
        "params": {"qubits": "1,121", "topology": "heavy-hex", "temp": "15", "stage": "5", "fridge_weight": "1.8", "code": "rotated", "physical_per": "17", "architect": "Chen", "fidelity": "99.5", "ops": "2.4", "coherence": "300", "speedup": "100", "variables": "127", "engineers": "450", "dev_years": "7"},
        "questions": [
            {"q": "How many qubits does the Condor processor have?", "a": "1,121"},
            {"q": "What is the two-qubit gate fidelity?", "a": "99.5 percent"},
            {"q": "What is the coherence time?", "a": "300 microseconds"},
            {"q": "How many engineers worked on the project?", "a": "450"},
        ],
    },
]

COUNT_TARGETS = ["the", "of", "and", "in", "to"]

random.seed(456)
data = []
task_id = 0

for pt in PASSAGE_TEMPLATES:
    passage = pt["text"].format(**pt["params"])

    for qi, qa in enumerate(pt["questions"]):
        # --- SINGLE TASK: comprehension only ---
        single_comp_prompt = (
            f"Read the following passage and answer the question.\n\n"
            f"{passage}\n\n"
            f"Question: {qa['q']}\n\n"
            f"Respond in this format:\nAnswer: <your answer>"
        )
        data.append({
            "task_id": task_id,
            "prompt": single_comp_prompt,
            "expected_answer": qa["a"],
            "expected_count": -1,  # no counting task
            "condition": "single_comprehension",
            "passage_id": pt["id"],
            "question_idx": qi,
            "count_target": "",
        })
        task_id += 1

    # --- SINGLE TASK: counting only ---
    for target in COUNT_TARGETS[:2]:  # 2 targets per passage
        actual_count = passage.lower().split(f" {target} ").__len__() - 1
        # more accurate count
        actual_count = len(re.findall(rf"\b{target}\b", passage.lower()))

        single_count_prompt = (
            f"Read the following passage and count every occurrence of the word \"{target}\".\n\n"
            f"{passage}\n\n"
            f"Respond in this format:\nCount: <number>"
        )
        data.append({
            "task_id": task_id,
            "prompt": single_count_prompt,
            "expected_answer": "",
            "expected_count": actual_count,
            "condition": "single_counting",
            "passage_id": pt["id"],
            "question_idx": -1,
            "count_target": target,
        })
        task_id += 1

    # --- DUAL TASK: comprehension + counting ---
    for qi, qa in enumerate(pt["questions"]):
        for target in COUNT_TARGETS[:2]:
            actual_count = len(re.findall(rf"\b{target}\b", passage.lower()))

            dual_prompt = (
                f"Read the following passage. You must do TWO things:\n"
                f"1. Count every occurrence of the word \"{target}\"\n"
                f"2. Answer the comprehension question at the end\n\n"
                f"{passage}\n\n"
                f"Question: {qa['q']}\n\n"
                f"Respond in EXACTLY this format:\n"
                f"Count: <number of times \"{target}\" appears>\n"
                f"Answer: <your answer to the question>"
            )
            data.append({
                "task_id": task_id,
                "prompt": dual_prompt,
                "expected_answer": qa["a"],
                "expected_count": actual_count,
                "condition": "dual",
                "passage_id": pt["id"],
                "question_idx": qi,
                "count_target": target,
            })
            task_id += 1

df_all = pd.DataFrame(data)
print(f"Generated {len(df_all)} items")
print(f"By condition: {dict(df_all['condition'].value_counts())}")

## Task Definition

In [ ]:
@kbench.task(
    name="dual_task_interference",
    description="Concurrent comprehension + counting — measures dual-task cost and asymmetric interference"
)
def dual_task_interference(
    llm,
    prompt: str,
    expected_answer: str,
    expected_count: int,
    condition: str,
) -> tuple[int, int]:
    response = llm.prompt(prompt)

    correct = 0
    total = 0

    if condition == "single_comprehension":
        # Only score comprehension
        total = 1
        answer = extract_answer(response)
        exp_norm = expected_answer.lower().strip()
        ans_norm = answer.lower().strip()
        if exp_norm in ans_norm or ans_norm in exp_norm:
            correct = 1
        else:
            # Number normalization
            exp_d = re.sub(r"[,\s]", "", exp_norm)
            ans_d = re.sub(r"[,\s]", "", ans_norm)
            if exp_d == ans_d and exp_d:
                correct = 1

    elif condition == "single_counting":
        # Only score counting
        total = 1
        count = extract_count(response)
        if count is not None and abs(count - int(expected_count)) <= 1:  # ±1 tolerance
            correct = 1

    elif condition == "dual":
        # Score both
        total = 2
        # Comprehension
        answer = extract_answer(response)
        exp_norm = expected_answer.lower().strip()
        ans_norm = answer.lower().strip()
        if exp_norm in ans_norm or ans_norm in exp_norm:
            correct += 1
        else:
            exp_d = re.sub(r"[,\s]", "", exp_norm)
            ans_d = re.sub(r"[,\s]", "", ans_norm)
            if exp_d == ans_d and exp_d:
                correct += 1
        # Counting
        count = extract_count(response)
        if count is not None and abs(count - int(expected_count)) <= 1:
            correct += 1

    return (correct, total)

## Run Evaluation

In [ ]:
eval_df = df_all[["prompt", "expected_answer", "expected_count", "condition"]].copy()

print(f"Running {len(eval_df)} items with kbench.llm...")
runs = dual_task_interference.evaluate(
    llm=kbench.llm,
    evaluation_data=eval_df,
    n_jobs=2,
    timeout=120,
    max_attempts=2,
)
results = runs.as_dataframe()
results["accuracy"] = results["result"].apply(
    lambda r: r[0] / r[1] if isinstance(r, tuple) and r[1] > 0 else float(r) if not isinstance(r, tuple) else 0
)
results = results.merge(
    df_all[["prompt", "condition", "passage_id", "question_idx", "count_target"]],
    on="prompt", how="left", suffixes=("", "_meta")
)
print(f"Collected {len(results)} results")

## Results & Analysis

In [ ]:
print("=== Accuracy by Condition ===")
cond_stats = results.groupby("condition")["accuracy"].agg(["mean", "std", "count"])
print(cond_stats.to_string(float_format="%.3f"))

single_comp = results[results["condition"] == "single_comprehension"]["accuracy"].mean()
single_count = results[results["condition"] == "single_counting"]["accuracy"].mean()
dual = results[results["condition"] == "dual"]["accuracy"].mean()

print(f"\n=== Dual-Task Costs ===")
print(f"Single comprehension:  {single_comp:.2%}")
print(f"Single counting:       {single_count:.2%}")
print(f"Dual-task (combined):  {dual:.2%}")
if single_comp > 0:
    comp_cost = single_comp - dual
    print(f"Dual-task cost:        {comp_cost:.2%} drop from single-task baseline")

print(f"\n=== By Passage ===")
print(results.groupby(["passage_id", "condition"])["accuracy"].mean().unstack().to_string(float_format="%.2f"))

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
conditions = ["single_comprehension", "single_counting", "dual"]
labels = ["Comprehension\n(single)", "Counting\n(single)", "Both\n(dual)"]
colors = ["steelblue", "darkorange", "seagreen"]

means = [results[results["condition"] == c]["accuracy"].mean() for c in conditions]
stds = [results[results["condition"] == c]["accuracy"].std() for c in conditions]

bars = ax.bar(labels, means, yerr=stds, color=colors, capsize=5, edgecolor="black", linewidth=0.5)
ax.set_ylabel("Accuracy")
ax.set_title("Dual-Task Interference: Single vs. Concurrent Performance")
ax.set_ylim(0, 1.15)
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03, f"{mean:.0%}",
            ha="center", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("dual_task_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to dual_task_results.png")